## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 💹 Azure AI Agent with Code Interpreter - Financial Analytics 📊

This notebook demonstrates using `FoundryChatClient` + `Agent` with Code Interpreter for use cases like portfolio analysis, compound interest calculations, and loan amortization.

## Features Covered:
- Setting up a FoundryChatClient with CodeInterpreterTool
- Executing Python code for portfolio analysis
- Compound interest and loan amortization calculations
- Streaming and non-streaming responses

### ⚠️ Important Financial Disclaimer ⚠️
> **The financial calculations provided by this notebook are for educational purposes only. Always verify calculations with qualified financial professionals before making financial decisions.**

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Project**: Access to an Microsoft Foundry project with deployed models
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
4. **Dependencies**: Required agent-framework packages installed

If you need to use a different tenant, specify the tenant ID:
```bash
az login --tenant <tenant-id>
```

## Import Libraries

Import the required libraries using the `FoundryChatClient` + `Agent` API with `CodeInterpreterTool`:

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import sys
from importlib.metadata import version

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print({p: version(p) for p in ("agent-framework-core", "agent-framework-foundry", "azure-ai-projects")})

## Initial Setup

Load environment variables from the `.env` file:

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

# Resolve the root from either the notebook directory or repository directory.
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.in").is_file())
load_dotenv(repo_root / ".env", override=False)
endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not endpoint or not model:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), "Select the repository .venv kernel."
print("Project endpoint and model: configured (values hidden)")

## Create Code Interpreter Tool 🛠️

Use the `FoundryChatClient.get_code_interpreter_tool()` factory method to create a `CodeInterpreterTool` that enables the agent to write and execute Python code in a secure sandbox:

In [ ]:
# Create a CodeInterpreterTool using the FoundryChatClient factory method
code_interpreter_tool = FoundryChatClient.get_code_interpreter_tool()
print(f"✅ Created CodeInterpreterTool: {type(code_interpreter_tool).__name__}")

## Create and Run the Financial Analytics Agent 💹

Now let's create a Financial Analytics agent using `FoundryChatClient` + `Agent` with the `CodeInterpreterTool` to write and execute Python code for financial calculations:

In [ ]:
async def main() -> None:
    """Use FoundryChatClient + Agent with Code Interpreter for financial analytics."""
    print("=== 💹 Azure AI Financial Analytics Agent with Code Interpreter ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="FinancialAnalyticsAgent",
                instructions=(
                    "You are a Financial Analytics expert who can write and execute Python code to perform "
                    "financial calculations including compound interest, loan amortization, portfolio analysis, "
                    "and investment projections. Always show your calculations clearly."
                ),
                tools=[code_interpreter_tool],
            )
            query = (
                "Calculate the compound interest for an investment of $10,000 at 7% annual rate "
                "compounded monthly for 5 years. Show the Python code and execute it to display the "
                "final value and total interest earned."
            )
            print(f"🤔 User: {query}")
            async with asyncio.timeout(120):
                response = await agent.run(query)
            assert response.text, "The service returned no answer."
            print(f"💹 Agent: {response.text}")
        finally:
            await client.client.close()
            await client.project_client.close()

## Execute the Example 🚀

Run the main function to see the Financial Analytics agent in action:

In [ ]:
# Run the main function
await main()

## Loan Amortization Analysis Example 📊

Let's create an example that calculates a loan amortization schedule:

In [ ]:
async def loan_amortization_example() -> None:
    """Calculate a loan amortization schedule with the code interpreter."""
    print("=== 📊 Loan Amortization Analysis Example ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="LoanAnalyticsAgent",
                instructions=(
                    "You are a Loan Analytics expert who can write and execute Python code to calculate "
                    "loan amortization schedules, monthly payments, and total interest costs. "
                    "Always explain your approach before writing code and create visualizations when helpful."
                ),
                tools=[code_interpreter_tool],
            )
            query = (
                "Create a loan amortization schedule for a $300,000 mortgage at 6.5% APR for 30 years. "
                "Show me the monthly payment, create a table showing the first 12 months of payments "
                "(principal vs interest breakdown), and calculate total interest paid over the life of the loan."
            )
            print(f"🤔 User: {query}")
            async with asyncio.timeout(120):
                response = await agent.run(query)
            assert response.text, "The service returned no answer."
            print(f"📊 Agent: {response.text}")
        finally:
            await client.client.close()
            await client.project_client.close()


await loan_amortization_example()

## Portfolio Analysis Example 💼

Test the agent with comprehensive portfolio analysis tasks:

In [ ]:
async def portfolio_analysis_examples() -> None:
    """Run several independent financial analysis tasks with the code interpreter."""
    print("=== 💼 Portfolio Analysis Examples ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="PortfolioAnalystAgent",
                instructions=(
                    "You are an expert Financial Analyst specializing in portfolio analysis and risk assessment. "
                    "Write clean, efficient Python code and explain your analysis clearly. "
                    "Create visualizations when they help illustrate financial concepts."
                ),
                tools=[code_interpreter_tool],
            )
            queries = [
                "Calculate the future value of monthly $500 investments over 20 years at 8% annual return (SIP/401k simulation)",
                "Create a simple portfolio risk analysis: If I have 60% stocks (expected return 10%, std dev 20%) and 40% bonds (expected return 4%, std dev 5%), what's the portfolio expected return and risk?",
                "Calculate the break-even point for a business with fixed costs of $50,000, variable cost per unit of $25, and selling price of $75 per unit",
            ]
            for i, query in enumerate(queries, 1):
                print(f"\n{'='*60}")
                print(f"--- 💼 Financial Analysis {i} ---")
                print(f"🤔 Question: {query}")
                print("-" * 40)
                async with asyncio.timeout(120):
                    response = await agent.run(query)
                assert response.text, f"The service returned no answer for analysis {i}."
                print(f"📊 Analysis: {response.text}")
        finally:
            await client.client.close()
            await client.project_client.close()


await portfolio_analysis_examples()

## Key Takeaways 📚

### API Pattern

```python
# The code interpreter tool is created from the FoundryChatClient factory and attached to the agent
code_interpreter_tool = FoundryChatClient.get_code_interpreter_tool()

client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=AzureCliCredential())
agent = Agent(
    client=client,
    name="FinancialAnalyticsAgent",
    instructions="...",
    tools=[code_interpreter_tool],
)
response = await agent.run(query)
```

### Features

1. **Code Interpreter**: `FoundryChatClient.get_code_interpreter_tool()` returns a hosted `CodeInterpreterTool` that runs Python in a secure sandbox.
2. **Financial Calculations**: Ideal for compound interest, amortization, portfolio analysis, and risk assessment.
3. **Data Analysis**: Supports financial modeling including projections and what-if scenarios.
4. **Streaming Support**: Works with both `await agent.run(...)` and `agent.run(..., stream=True)`.

### Use Cases

- **Compound Interest Calculations**: Investment growth projections
- **Loan Amortization**: Mortgage and loan payment schedules
- **Portfolio Analysis**: Risk assessment and return calculations
- **Investment Projections**: SIP, 401k, retirement planning simulations
- **Break-even Analysis**: Business financial modeling

### Best Practices

1. **Clear Instructions**: Specify the type of financial calculations needed.
2. **Lifecycle**: Close the underlying OpenAI and project clients in a `finally` block.
3. **Bounded Calls**: Cloud operations are bounded with `asyncio.timeout(...)`.
4. **Disclaimers**: Include appropriate financial disclaimers for all calculations.

Validated against `agent-framework-core==1.17.0`.